# Pipeline de Ingesta: Construcción del Grafo de Conocimiento

Sistema Graph RAG sobre el canon de Sherlock Holmes.
Este notebook ejecuta el pipeline completo de ingesta:
1. Descarga de textos de Project Gutenberg
2. Separación en relatos individuales
3. Chunking consciente de la estructura
4. Extracción multipaso de entidades y relaciones
5. Entity resolution
6. Población del grafo en Neo4j

In [1]:
%load_ext autoreload
%autoreload 2
# Si ejecutas desde notebooks/, necesitas que graphrag sea importable.
# Con uv: uv sync --extra dev && pip install -e .
from graphrag.config import get_settings
from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.ingestion.text_processor import TextProcessor
from graphrag.ingestion.entity_extractor import EntityExtractor

settings = get_settings()
print(f"Proyecto GCP: {settings.google_cloud_project}")
print(f"Neo4j URI: {settings.neo4j_uri}")
print(f"Chunk size: {settings.chunk_size}")

Proyecto GCP: holmesgraphrag
Neo4j URI: bolt://localhost:7687
Chunk size: 1500


## 1. Inicializar Neo4j y crear esquema

In [2]:
neo4j = Neo4jManager()
neo4j.setup_database()  # Crea constraints + índices vectoriales + fulltext
print("Base de datos inicializada")
print(f"Stats actuales: {neo4j.get_stats()}")

Base de datos inicializada
Stats actuales: {'Chunk': 350, 'Story': 10}


## 2. Descargar y procesar textos de Gutenberg

In [3]:
processor = TextProcessor(neo4j_manager=neo4j)

#Solo los 10 relatos de desarrollo
story_chunks = processor.process_phase1()

print(f"\nRelatos procesados: {len(story_chunks)}")
for title, chunks in story_chunks.items():
    print(f"  - {title}: {len(chunks)} chunks")

Generando embeddings: 100%|██████████| 37/37 [00:00<00:00, 65.50it/s]                          
                                                                                      


Relatos procesados: 10
  - The Adventure Of The Dancing Men: 35 chunks
  - Silver Blaze: 36 chunks
  - The Final Problem: 27 chunks
  - A Scandal In Bohemia: 32 chunks
  - The Red-Headed League: 34 chunks
  - A Case Of Identity: 26 chunks
  - The Five Orange Pips: 27 chunks
  - The Adventure Of The Blue Carbuncle: 29 chunks
  - The Adventure Of The Speckled Band: 67 chunks
  - The Adventure Of The Copper Beeches: 37 chunks


## 3. Extracción de entidades y relaciones

Extracción multipaso con sliding context:
- **Paso 1**: Extracción de entidades (Characters, Locations, Crimes, Objects, Deductions, Scenes, Events)
- **Paso 2**: Extracción de relaciones entre las entidades encontradas
- **Entity Resolution**: Normalización + embeddings + LLM para desambiguar duplicados

In [4]:
'''
import logging
logging.getLogger('graphrag.ingestion.entity_extractor').setLevel(logging.DEBUG)

extractor = EntityExtractor()

all_results = {}
story_title = "A SCANDAL IN BOHEMIA"
chunks = story_chunks[story_title]

# Solo 3 chunks para debug rápido
result = extractor.process_story_chunks(chunks, story_title)
all_results[story_title] = result

entities = result["entities"]
print(f"Personajes: {len(entities.get('characters', []))}")
for c in entities["characters"]:
  print(f"  [{c['name']}] aliases: {c.get('aliases', [])}")
'''

'\nimport logging\nlogging.getLogger(\'graphrag.ingestion.entity_extractor\').setLevel(logging.DEBUG)\n\nextractor = EntityExtractor()\n\nall_results = {}\nstory_title = "A SCANDAL IN BOHEMIA"\nchunks = story_chunks[story_title]\n\n# Solo 3 chunks para debug rápido\nresult = extractor.process_story_chunks(chunks, story_title)\nall_results[story_title] = result\n\nentities = result["entities"]\nprint(f"Personajes: {len(entities.get(\'characters\', []))}")\nfor c in entities["characters"]:\n  print(f"  [{c[\'name\']}] aliases: {c.get(\'aliases\', [])}")\n'

In [5]:
extractor = EntityExtractor()

# Para prueba — quitar el slice para el run completo
TEST_STORIES = ["The Final Problem", "A Case Of Identity", "The Red-Headed League"]

all_results = {}
for story_title, chunks in story_chunks.items():
    if story_title not in TEST_STORIES:
        continue

    print(f"\n{'='*60}")
    print(f"Procesando: {story_title}")
    print(f"{'='*60}")

    result = extractor.process_story_chunks(chunks, story_title)
    all_results[story_title] = result

    # Resumen
    entities = result["entities"]
    n_chars = len(entities.get("characters", []))
    n_locs = len(entities.get("locations", []))
    n_crimes = len(entities.get("crimes", []))
    n_deductions = len(entities.get("deductions", []))
    print(f"  Personajes: {n_chars}, Ubicaciones: {n_locs}, Crímenes: {n_crimes}, Deducciones: {n_deductions}")
    print(f"  Relaciones: {len(result['relationships'])}")


Procesando: The Final Problem


Generando embeddings: 100%|██████████| 69/69 [00:03<00:00, 21.00it/s]


  Personajes: 12, Ubicaciones: 17, Crímenes: 7, Deducciones: 16
  Relaciones: 349

Procesando: The Red-Headed League


Generando embeddings: 100%|██████████| 81/81 [00:03<00:00, 23.20it/s]


  Personajes: 20, Ubicaciones: 26, Crímenes: 7, Deducciones: 26
  Relaciones: 535

Procesando: A Case Of Identity


Extrayendo 'A Case Of Identity':  31%|███       | 8/26 [08:30<18:25, 61.43s/it] Error transitorio (intento 1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Reintentando en 5s.
Error transitorio (intento 2/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Reintentando en 10s.
Error transitorio (intento 3/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Reintentando en 20s.
Error extrayendo relaciones: 4

  Personajes: 18, Ubicaciones: 6, Crímenes: 7, Deducciones: 22
  Relaciones: 283


In [6]:
import json
import os

# Guarda los resultados de extracción a disco por si el pipeline se interrumpe.
# Para recargar sin re-extraer: all_results = json.load(open("../output/extraction_results.json"))
os.makedirs("../output", exist_ok=True)
with open("../output/extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"Checkpoint guardado: ../output/extraction_results.json ({len(all_results)} relatos)")

Checkpoint guardado: ../output/extraction_results.json (3 relatos)


In [7]:
# Resolución cross-story: unifica nombres canónicos entre relatos.
# Garantiza que Holmes y Watson tengan el mismo nombre canónico en Neo4j
# independientemente del relato de origen.
all_results = extractor.normalize_cross_story_entities(all_results)

# Guarda el checkpoint normalizado (sobreescribe el anterior)
with open("../output/extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("Cross-story normalization completada.")
# Verificación rápida
for story, result in all_results.items():
    chars = result["entities"].get("characters", [])
    holmes = next((c["name"] for c in chars if "holmes" in c["name"].lower()), "--")
    watson = next((c["name"] for c in chars if "watson" in c["name"].lower()), "--")
    print(f"  {story[:40]:40s}  Holmes='{holmes}'  Watson='{watson}'")

Cross-story normalization completada.
  The Final Problem                         Holmes='Mr. Sherlock Holmes'  Watson='Dr. Watson'
  The Red-Headed League                     Holmes='Mr. Sherlock Holmes'  Watson='Dr. Watson'
  A Case Of Identity                        Holmes='Mr. Sherlock Holmes'  Watson='Dr. Watson'


## 4. Poblar el grafo en Neo4j

In [8]:
for story_title, result in all_results.items():
    print(f"Almacenando: {story_title}")

    # Almacenar entidades
    neo4j.store_entities(result["entities"], story_title)

    # Almacenar relaciones
    neo4j.store_relationships(result["relationships"], story_title=story_title)

    # Vincular chunks con las entidades que mencionan
    for chunk_info in result.get("chunk_entities", []):
        if chunk_info["chunk_id"] and chunk_info["entity_names"]:
            neo4j.link_chunk_to_entities(chunk_info["chunk_id"], chunk_info["entity_names"])

print("\nGrafo poblado exitosamente")

Almacenando: The Final Problem
Almacenando: The Red-Headed League
Almacenando: A Case Of Identity

Grafo poblado exitosamente


## 5. Verificar el grafo

In [9]:
stats = neo4j.get_stats()
print("Estadísticas del grafo:")
for label, count in stats.items():
    print(f"  {label}: {count}")

Estadísticas del grafo:
  Chunk: 350
  Event: 124
  Object: 123
  Scene: 87
  Deduction: 64
  Location: 49
  Character: 45
  Crime: 21
  Story: 10


In [10]:
# Ver personajes más conectados
top_characters = neo4j.execute_query("""
MATCH (c:Character)-[r]-()
RETURN c.name AS name, count(DISTINCT r) AS connections
ORDER BY connections DESC
LIMIT 10
""")

print("\nPersonajes más conectados:")
for char in top_characters:
    print(f"  {char['name']}: {char['connections']} conexiones")


Personajes más conectados:
  Mr. Sherlock Holmes: 216 conexiones
  Dr. Watson: 106 conexiones
  Mr. Jabez Wilson: 57 conexiones
  Mr. Hosmer Angel: 33 conexiones
  Mr. James Windibank: 29 conexiones
  Mr. Merryweather: 28 conexiones
  Miss Mary Sutherland: 28 conexiones
  Colonel James Moriarty: 25 conexiones
  John Clay: 25 conexiones
  Vincent Spaulding: 23 conexiones


In [11]:
# Ver relatos y sus entidades
stories = neo4j.execute_query("""
MATCH (s:Story)
OPTIONAL MATCH (c:Character)-[:APPEARS_IN]->(s)
RETURN s.title AS story, s.collection AS collection, count(c) AS characters
ORDER BY characters DESC
""")

print("\nRelatos cargados:")
for s in stories:
    print(f"  {s['story']} ({s['collection']}): {s['characters']} personajes")


Relatos cargados:
  The Red-Headed League (The Adventures of Sherlock Holmes): 20 personajes
  A Case Of Identity (The Adventures of Sherlock Holmes): 18 personajes
  The Final Problem (The Memoirs of Sherlock Holmes): 12 personajes
  A Scandal In Bohemia (The Adventures of Sherlock Holmes): 0 personajes
  The Five Orange Pips (The Adventures of Sherlock Holmes): 0 personajes
  The Adventure Of The Blue Carbuncle (The Adventures of Sherlock Holmes): 0 personajes
  The Adventure Of The Speckled Band (The Adventures of Sherlock Holmes): 0 personajes
  The Adventure Of The Copper Beeches (The Adventures of Sherlock Holmes): 0 personajes
  Silver Blaze (The Memoirs of Sherlock Holmes): 0 personajes
  The Adventure Of The Dancing Men (The Return of Sherlock Holmes): 0 personajes


In [12]:
# Ver cadenas de deducción
deductions = neo4j.execute_query("""
MATCH (d:Deduction)-[:LEADS_TO]->(d2:Deduction)
RETURN d.observation AS from_obs, d2.observation AS to_obs
LIMIT 5
""")

if deductions:
    print("\nCadenas de deducción encontradas:")
    for d in deductions:
        print(f"  {d['from_obs'][:60]}... → {d['to_obs'][:60]}...")
else:
    print("\nNo se encontraron cadenas de deducción (LEADS_TO)")


Cadenas de deducción encontradas:
  Holmes falls upon his knees upon the floor and, with the lan... → Holmes has completed his examination of the floor and puts h...
  The assistant having come for half wages... → The man’s business was a small one, and there was nothing in...
  The man’s business was a small one, and there was nothing in... → I thought of the assistant’s fondness for photography, and h...
  I thought of the assistant’s fondness for photography, and h... → I made inquiries as to this mysterious assistant and found t...
  beating upon the pavement with my stick. I was ascertaining ... → His knees were what I wished to see. You must yourself have ...


## 6. Cleanup (opcional)

In [13]:
 # Descomentar para limpiar la base de datos completa
#neo4j.clear_database()
#print("Base de datos limpiada")

neo4j.close()
print("Conexión cerrada")

Conexión cerrada
